In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table
import os

print(os.getcwd())

path_old1 = '../data/raw/Multiwave_6_12_26.fits'
path_old2 = '../data/raw/Multiwave_UMAP_6_22_26.csv'
path = '../data/raw/Multiwave_UMAP_6_26_26.csv'

""" 
with fits.open(path) as hdul:
    table = Table(hdul[1].data)
    df = table.to_pandas()

"""

# ['Unnamed: 0.1', 'Unnamed: 0', 'name', 'non_agn', 'zErr_normgal', 'z_code_normgal', 
# 'opt_r', 'r_code_normgal', 'opt_g', 'g_code_normgal']

df = pd.read_csv(path)

df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df = df.set_index('name')
df = df.rename({'IR_agn': 'ir_agn'}, axis=1)

display(df.shape)

# New columns between first and last datasets
new_col = ['non_agn', 'zErr_normgal', 'z_code_normgal', 'opt_r', 'r_code_normgal', 'opt_g', 'g_code_normgal', 'CSC_b_lum']


bool_cols = ['conf_flag', 'extent_flag', 'xray_agn', 'optical_agn', 'ir_agn']
agn_cols = ['xray_agn', 'optical_agn', 'ir_agn']


# Fixing bool columns
df[bool_cols] = df[bool_cols].fillna(False).astype('bool')


# Treating ' ' as NaN
object_cols = df.select_dtypes(include='object').columns

df[object_cols] = df[object_cols].astype('string').apply(lambda col: col.str.strip())
df[object_cols] = df[object_cols].replace("", np.nan)


print(sorted(df.columns.tolist()))

# List of columns by dtype
for i in df.dtypes.unique():
    print(f'Columns with {i} dtype:', df.select_dtypes(include=i).columns.tolist())

# Missing value proportion DataFrame
missing = pd.DataFrame({
    'NaN perc': (df.isna().sum() / df.shape[0] * 100).round(2), 
    'missing cnt': df.isna().sum(), 
    'non-missing cnt': df.notna().sum(),
    }).sort_values('NaN perc', ascending=False)

for i in range(0, 100, 25):
    print(f'Columns with NaN values percentage between {i} and {i + 25}: ', end="") 
    print(missing[(missing['NaN perc'] >= i) & 
          (missing['NaN perc'] < i + 25)].shape[0])

missing

In [ ]:
# Statistical info about NaN values inside rows
missing_rows = df.isna().sum(axis=1)/df.shape[1] * 100
display(missing_rows.describe())


# Distribution of NaN percentages inside rows
print('Missing row distribution:')

for i in range(0, 100, 10):
    row_cnt = ((missing_rows >= i) & (missing_rows < i + 10)).sum()
    perc = row_cnt / df.shape[0] * 100
    print(f'Percentage of rows with missing values between {i} and {i + 10}: ', round(perc, 5))

In [ ]:
agn_cols = ["xray_agn", "optical_agn", "ir_agn"]
neg_col = "non_agn"

for col in agn_cols:
    if col in df.columns:
        print("\n")
        print(df[col].value_counts(dropna=False))

"""
for col in agn_cols:
    if col not in df.columns:
        print(f"Warning: {col} not found")
"""

df["agn_class"] = (
    df["xray_agn"].fillna(False).astype(bool) |
    df["optical_agn"].fillna(False).astype(bool) |
    df["ir_agn"].fillna(False).astype(bool)
)

print("\n", df["agn_class"].value_counts(dropna=False))

In [ ]:
# Augmenting features

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


plt.figure(figsize=(8.5,8.5))
plt.rcParams.update({'font.size': 15})

df_z_w = df.dropna(subset=['WISE_w1mpro', 'Rec_Z'])

# Adding WISE / IR features
df['WISE_W1_W2'] = df['WISE_w1mpro'] - df['WISE_w2mpro']
df['WISE_W2_W3'] = df['WISE_w2mpro'] - df['WISE_w3mpro']

# Stern criterion
# df["wise_stern"] = df["WISE_W1_W2"] >= 0.8

# Mateos wedge boundaries
x = df['WISE_W2_W3']
y = df['WISE_W1_W2']

mateos_wedge_mask = (
    (y <= 0.315*x + 0.796) &
    (y >= 0.315*x - 0.222) &
    (y >= -3.172*x + 7.264)
)

df["mateos_wedge"] = mateos_wedge_mask
print(df["mateos_wedge"].value_counts())

'''
These are the WISE magnitudes we subtract to get colors. Colors in astronomy refer to magnitudes or fluxes in two bands being subtracted from each other.
The WISE color-color diagram is in the IR regime.
'''


# optical colors
df["PS_g_r"] = df["PS21P_g"] - df["PS21P_r"]
df["PS_r_i"] = df["PS21P_r"] - df["PS21P_i"]

df["LEG_G_R"] = df["LEG21P_G"] - df["LEG21P_R"]
df["LEG_R_Z"] = df["LEG21P_R"] - df["LEG21P_Z"]


# optical-to-IR colors
df["PS_r_W1"] = df["PS21P_r"] - df["WISE_w1mpro"]
df["PS_i_W2"] = df["PS21P_i"] - df["WISE_w2mpro"]


# X-ray flux / optical flux relation
for col in ["flux_aper_b", "flux_aper_h", "flux_aper_s"]:
    df[f"log_{col}"] = np.log10(df[col].where(df[col] > 0))

df["xray_opt_r_relation"] = df["log_flux_aper_b"] + 0.4 * df["PS21P_r"]
df["xray_opt_g_relation"] = df["log_flux_aper_b"] + 0.4 * df["PS21P_g"]



print(df.describe())



'''
Defining the IR AGN in our subset, I'd recommend doing something like db['lum_sub']=db['name'].isin(db_sub['name']), which will return true if it's 
contained in the subset which makes for easy filtering
'''

# luminosity subset
lum_mask = (df['CSC_b_lum']>=1e42) & (df['CSC_b_lum']<=1e43)
df_sub = df[lum_mask]
print(r'$1e42<=L_{X}<=1e43:', len(df_sub))

x_sub = df_sub['WISE_W2_W3']
y_sub = df_sub['WISE_W1_W2']

wedge_mask_sub = (
    (y_sub <= 0.315*x_sub + 0.796) &
    (y_sub >= 0.315*x_sub - 0.222) &
    (y_sub >= -3.172*x_sub + 7.264)
)

df_sub['inside_wise_wedge'] = wedge_mask_sub


dub_sub_ir=df_sub[df_sub['inside_wise_wedge']]
print('IR AGN in subset:', len(dub_sub_ir))

#Physical boundaries of the wedge, easier to plot the endpoints than the actual function
plt.plot([1.855, 8.587], [1.380, 3.501], c='black', linestyle='--', linewidth=3.5)
plt.plot([2.147, 8.587], [0.454, 2.483], c='black', linestyle='--', linewidth=3.5)
plt.plot([1.855, 2.147], [1.380, 0.454], c='black', linestyle='--', linewidth=3.5)


plt.scatter(df_z_w['WISE_w2mpro']-df_z_w['WISE_w3mpro'], df_z_w['WISE_w1mpro']-df_z_w['WISE_w2mpro'], color='gray', alpha=0.7, label='Has z & WISE')
plt.scatter(x_sub, y_sub, color='salmon', alpha=0.85, label=(r'$1e42<=L_{X}<=1e43'))
plt.xlabel('W2-W3 (Vega)')
plt.ylabel('W1-W2 (Vega)')
plt.title('Wise Color-Color Diagram for ' + r'$1e42<=L_{X}<=1e43:')
plt.legend(loc='upper left')
plt.xlim(-1, 6.5)
plt.ylim(-1.15, 3.1)

In [ ]:
# Correction extinction
check_cor = [c for c in df.columns if any(k in c.lower() for k in [
    "ebv", "ext", "redd", "mw_trans", "trans", "flux"
])]

print(list(df[check_cor]))

# 1. Найти все возможные optical / IR / extinction-related columns
keywords = [
    "PS", "LEG", "WISE", "SDSS", "GAIA", "2MASS",
    "EBV", "E_BV", "EXT", "REDD", "DUST",
    "MW", "TRANS", "TRANSMISSION",
    "FLUX_G", "FLUX_R", "FLUX_Z",
    "MAG", "DERED"
]

cols_found = [
    c for c in df.columns
    if any(k.lower() in c.lower() for k in keywords)
]

print("Found columns:", len(cols_found))
for c in cols_found:
    print(c)
    
# 2. Проверка, какие Legacy и PS columns реально есть
groups = {
    "Pan-STARRS": ["PS21P_g", "PS21P_r", "PS21P_i", "PS21P_z", "PS21P_y"],
    "Legacy mags": ["LEG21P_G", "LEG21P_R", "LEG21P_Z"],
    "Legacy flux": ["FLUX_G", "FLUX_R", "FLUX_Z", "flux_g", "flux_r", "flux_z"],
    "Legacy MW transmission": [
        "MW_TRANSMISSION_G", "MW_TRANSMISSION_R", "MW_TRANSMISSION_Z",
        "mw_transmission_g", "mw_transmission_r", "mw_transmission_z"
    ],
    "Extinction": ["EBV", "ebv", "E_BV", "SFD_EBV", "extinction_g", "extinction_r", "extinction_z"]
}

for name, cols in groups.items():
    present = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    print(f"\n{name}")
    print("present:", present)
    print("missing:", missing)
    
    
all_columns = df.columns
for name in ["PS21P_g", "PS21P_r", "PS21P_i", "LEG21P_G", "LEG21P_R", "LEG21P_Z", "WISE_w1mpro", "WISE_w2mpro"]:
    if name in all_columns.names:
        idx = all_columns.names.index(name)
        print("\n", name)
        print(all_columns[idx])
        
        
df.loc[:, ["ra","dec"]]

In [ ]:
# Если колонки уже bool, это ок. Если там строки типа "False"/"True", безопасно приводим.
def to_bool_safe(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    return (
        s.astype("string")
        .str.strip()
        .str.lower()
        .isin(["true", "t", "1", "yes", "y"])
    )

for col in agn_cols + [neg_col]:
    df[col] = to_bool_safe(df[col])

# 1. Positive marker: confirmed AGN
df["agn_positive"] = df[agn_cols].any(axis=1)

# 2. Negative marker: confirmed non-AGN / galaxies
df["agn_negative"] = df[neg_col]

# 3. Conflict marker: if object is simultaneously marked as AGN and non-AGN
df["label_conflict"] = df["agn_positive"] & df["agn_negative"]
print("Label conflicts:", (df["agn_positive"] & df["agn_negative"]).sum())

df["agn_negative"] = df["agn_negative"] & ~df["agn_positive"] # priority to positive labels


# 4. Silver score
df["silver_score"] = 0.0

# WISE color
df.loc[df["mateos_wedge"].fillna(False), "silver_score"] += 0.35
# df.loc[df["wise_stern"].fillna(False), "silver_score"] += 0.25


# High X-ray-to-optical proxy
q80_r = df["xray_opt_r_relation"].quantile(0.80) 
df.loc[df["xray_opt_r_relation"] >= q80_r, "silver_score"] += 0.15

# X-ray luminosity
# df.loc[df["CSC_b_lum"].between(1e42, 1e43), "silver_score"] += 0.20

# Finalizing
df["silver_score"] = df["silver_score"].clip(0, 1)
df["silver_agn"] = (
    (df["silver_score"] >= 0.01) &
    ~df["agn_positive"] &
    ~df["agn_negative"]
)

print(df["silver_agn"].value_counts())


# UMAP
df["label_3class"] = np.select(
    [
        df["agn_positive"],
        df["agn_negative"],
        df["silver_agn"]
    ],
    [
        2,   # confirmed AGN
        0,   # confirmed non-AGN
        1,   # AGNs from silver dataset
    ],
    default=-1  # unlabeled
).astype("int8")

# Defragmentation of dataset
df = df.copy()
df_umap_base = df.copy()



# ML 
df["label_source"] = "unlabeled"

df.loc[df["agn_negative"], "label_source"] = "negative"
df.loc[df["silver_agn"], "label_source"] = "silver"
df.loc[df["agn_positive"], "label_source"] = "gold"

df["train_label"] = -1

df.loc[df["agn_negative"], "train_label"] = 0
df.loc[df["silver_agn"], "train_label"] = 1
df.loc[df["agn_positive"], "train_label"] = 1

df["sample_weight"] = 0.

df.loc[df["agn_negative"], "sample_weight"] = 1.
df.loc[df["agn_positive"], "sample_weight"] = 1.
df.loc[df["silver_agn"], "sample_weight"] = (
    0.3 +
    0.7*df.loc[df["silver_agn"], "silver_score"]
)


# Defragmentation of dataset
df = df.copy()
df_train = df.copy()

print(df["label_3class"].value_counts().sort_index())
print(df["label_source"].value_counts())
#anchor1

In [ ]:
for col in df.columns:
    print(col, end=' ')
    
df.shape

# UMAP Part

We prepare data, perform PCA to reduce noise and unnecesarry columns, and then plot it against data, NaN values, observations by source, etc.

# Reduced Dataset Case

*!* Run this cell only if we need to do UMAP on reduced (without NaN values) subset

In [ ]:
display(df_umap_base.shape)

df_umap_base['combo_g']=df_umap_base['PS21P_g'].combine_first(df_umap_base['LEG21P_G'])
df_umap_base['combo_r']=df_umap_base['PS21P_r'].combine_first(df_umap_base['LEG21P_R'])
df_umap_base['combo_z_mag']=df_umap_base['PS21P_z'].combine_first(df_umap_base['LEG21P_Z'])


df_umap_base = df_umap_base.dropna(subset=['significance', 'flux_aper_b', 'flux_aper_lolim_b', 'flux_aper_hilim_b', 'flux_aper_h','flux_aper_lolim_h',
'flux_aper_hilim_h','flux_aper_s','flux_aper_lolim_s','flux_aper_hilim_s','flux_aper_m','flux_aper_lolim_m','flux_aper_hilim_m','combo_g','combo_r','combo_z_mag','WISE_w1mpro', 
'WISE_w1snr','WISE_w2mpro', 'WISE_w2snr','WISE_w3mpro', 'WISE_w3snr', "hard_hs", "hard_hs_lolim", "hard_hs_hilim"])

df_sample = df_umap_base.copy()

reduced_cols = ['significance', 'flux_aper_b', 'flux_aper_lolim_b', 'flux_aper_hilim_b', 'flux_aper_h','flux_aper_lolim_h',
'flux_aper_hilim_h','flux_aper_s','flux_aper_lolim_s','flux_aper_hilim_s','flux_aper_m','flux_aper_lolim_m','flux_aper_hilim_m','combo_g','combo_r','combo_z_mag','WISE_w1mpro', 
'WISE_w1snr','WISE_w2mpro', 'WISE_w2snr','WISE_w3mpro', 'WISE_w3snr', "hard_hs", "hard_hs_lolim", "hard_hs_hilim"]



df_umap_base = df_umap_base[reduced_cols]

display(df_umap_base.shape)

X_pca = df_umap_base.copy()


# General UMAP case

For general case, we avoid previous column, and run all the cells except previous one

In [ ]:
exclude_cols = [
    "name",
    "col1",
    
    # AGN labels
    "xray_agn",
    "optical_agn",
    "ir_agn",
    "agn_class",
    "agn_positive",
    
    # non-AGN labels
    "non_agn",
    "agn_negative",
    
    # derived labels
    "label_3class",
    "label_conflict",
    "silver_score",
    "silver_agn",
    "binary_target",
    
    # training info
    "train_label", "train_weight", "label_source", "sample_weight",
    
    # masks
    "mateos_wedge",
    "wise_stern",

    # class / metadata columns
    "likelihood_class",
    "SDSS_CLASS",
    "SDSS_SUBCLASS",
    "2MASS21P_designation"
]

exclude_cols = [c for c in exclude_cols if c in df_umap_base.columns]

X = df_umap_base.drop(columns=exclude_cols, errors="ignore")
print(X.shape)

X = X.select_dtypes(include=[np.number]).copy()
print(X.shape)


display(X.head(20))

In [ ]:
# Dropping sparse features
feature_missing_percent = X.isna().mean() * 100

display(
    feature_missing_percent.sort_values(ascending=False).head(10)
)

max_feature_missing = 90

keep_cols = feature_missing_percent[feature_missing_percent <= max_feature_missing].index
X = X[keep_cols].copy()

print(X.shape)

In [ ]:
# Special column processing (big data): ['CSC_b_lum']
num = X.select_dtypes(include=[np.number])
summary = pd.DataFrame({
    "dtype": num.dtypes,
    "missing_pct": num.isna().mean() * 100,
    "min": num[num != 0].min(),
    "max": num.max(),
    "max_abs": num.abs().max(),
    "median": num.median(),
    "mean": num.mean(),
    "std": num.std(),
    "nunique": num.nunique()
}).sort_values("max_abs", ascending=False)

display(summary.head(50))


big_cols = summary[summary["max_abs"] > 1e20].index.tolist()
display(big_cols)

for col in big_cols: 
    X.loc[X[col] <= 0, col] = np.nan
    X[col + "_log10"] = np.log10(X[col])
    X = X.drop(columns=col)

X = X.copy()

In [ ]:
# Imputing and encoding missing data
from sklearn.impute import SimpleImputer

sample_size = len(df_umap_base) # we can adjust however we want

df_sample = df_umap_base.sample(
    n=min(sample_size, len(df_umap_base)),
    random_state=42
).sort_index().copy()

X_sample = X.loc[df_sample.index].copy()

print(df_sample.shape)
print(X_sample.shape)



missing_indicators = X_sample.isna().astype("int8")
missing_indicators.columns = [f"{col}_missing" for col in X_sample.columns]

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_sample)


X_umap_input = np.hstack([
    X_imputed,
    missing_indicators.values
]).astype("float32")

print(X_umap_input.shape)

In [ ]:
# PCA before UMAP
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# We can also use RobustScaler (instead of StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_umap_input).astype("float32")

# Checking most optimal parameters
for n in [10, 20, 30, 50, 100]: 
    pca = PCA(n_components=n, random_state=42)
    X_pca = pca.fit_transform(X_scaled).astype("float32")

    print(
        n,
        pca.explained_variance_ratio_.sum()
    )

n_pca = min(50, X_scaled.shape[1]) # we can adjust; max_n_cols = 216

pca = PCA(n_components=n_pca, random_state=42)
X_pca = pca.fit_transform(X_scaled).astype("float32")

print(X_pca.shape)
print("Explained variance:", pca.explained_variance_ratio_.sum())


In [ ]:
# UMAP
import umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    metric="euclidean",
    random_state=42,
    n_jobs=-1,
    low_memory=True,
)

embedding = reducer.fit_transform(X_pca)

print(embedding.shape)

umap_df = pd.DataFrame(
    {
        "UMAP1": embedding[:, 0],
        "UMAP2": embedding[:, 1]
    },
    index=df_sample.index
)

umap_df["label_3class"] = df_sample["label_3class"].values
umap_df["agn_positive"] = df_sample["agn_positive"].values
umap_df["agn_negative"] = df_sample["agn_negative"].values
umap_df["silver_agn"] = df_sample["silver_agn"].values
umap_df["significance"] = df_sample["significance"].values
umap_df["NaN_percent"] = df_sample.isna().mean(axis=1).values * 100

display(umap_df.head(20))

In [ ]:
# Plotting UMAP structure
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 8))

plt.scatter(
    umap_df["UMAP1"],
    umap_df["UMAP2"],
    s=2,
    alpha=0.5
)

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP projection of multiwavelength source features")
plt.show()

In [ ]:
plt.figure(figsize=(9, 8))

mask_unlabeled = umap_df["label_3class"] == -1
mask_significance = ((umap_df["significance"] >= 3) & (umap_df["label_3class"] == 0))
mask_nonagn = ((umap_df["label_3class"] == 0) & (umap_df["significance"] < 3))
mask_agn = umap_df["label_3class"] == 2
mask_silver = umap_df["label_3class"] == 1


plt.scatter(
    umap_df.loc[mask_unlabeled, "UMAP1"],
    umap_df.loc[mask_unlabeled, "UMAP2"],
    s=2,
    alpha=0.15,
    color="lightgray",
    label="Unlabeled"
)

plt.scatter(
    umap_df.loc[mask_nonagn, "UMAP1"],
    umap_df.loc[mask_nonagn, "UMAP2"],
    s=5,
    alpha=0.75,
    color="black",
    label="Confirmed non-AGN / galaxies"
)

plt.scatter(
    umap_df.loc[mask_significance , "UMAP1"],
    umap_df.loc[mask_significance, "UMAP2"],
    s=5,
    alpha=0.75,
    color="tab:blue",
    label="Galaxies with significance >= 3"
)

"""
plt.scatter(
    umap_df.loc[mask_agn, "UMAP1"],
    umap_df.loc[mask_agn, "UMAP2"],
    s=5,
    alpha=0.75,
    color="tab:",
    label="Confirmed AGN"
)
"""


# candidate_mask = df_train["score_bagged_pseudo_lgbm"] > 0.8
"""
plt.scatter(
    umap_df.loc[mask_silver, "UMAP1"],
    umap_df.loc[mask_silver, "UMAP2"],
    s=5,
    alpha=0.75,
    color="tab:orange",
    label="Objects from silver sample"
)
"""

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP: confirmed AGN, confirmed non-AGN, and unlabeled sources")
plt.legend(markerscale=2)
plt.legend(fontsize=10)
plt.show()

umap_df["label_3class"].value_counts()

In [ ]:
for label_col in ["xray_agn", "optical_agn", "ir_agn"]:
    if label_col in df.columns:
        label_values = df[label_col].fillna(False).astype(bool)
        plt.figure(figsize=(9, 8))

        plt.scatter(
            umap_df.loc[mask_unlabeled, "UMAP1"],
            umap_df.loc[mask_unlabeled, "UMAP2"],
            s=2,
            alpha=0.15,
            color="lightgray",
            label="Unlabeled"
        )

        """
        plt.scatter(
            umap_df.loc[mask_nonagn, "UMAP1"],
            umap_df.loc[mask_nonagn, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:blue",
            label="Confirmed non-AGN / galaxies"
        )
        """
        
        plt.scatter(
            umap_df.loc[label_values, "UMAP1"],
            umap_df.loc[label_values, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:red",
            label=f"{label_col} AGN"
        )

# variability flag to add (in infrared); 
# do corrections for formulas;
# ;

        plt.xlabel("UMAP1")
        plt.ylabel("UMAP2")
        plt.title("UMAP: confirmed AGN, confirmed non-AGN, and unlabeled sources")
        plt.legend(markerscale=2)
        plt.legend(fontsize=10)
        plt.show()

In [ ]:
plt.figure(figsize=(9, 8))

mask_unlabeled = umap_df["label_3class"] == -1
mask_nonagn = umap_df["label_3class"] == 0
mask_agn = umap_df["label_3class"] == 2
mask_silver = umap_df["label_3class"] == 1
candidate_mask = ((df_train["score_bagged_pseudo_lgbm"] > 0.8) & (~mask_agn))


plt.scatter(
    umap_df.loc[mask_unlabeled, "UMAP1"],
    umap_df.loc[mask_unlabeled, "UMAP2"],
    s=2,
    alpha=0.15,
    color="lightgray",
    label="Unlabeled"
)

plt.scatter(
    umap_df.loc[mask_nonagn, "UMAP1"],
    umap_df.loc[mask_nonagn, "UMAP2"],
    s=5,
    alpha=0.75,
    color="tab:blue",
    label="Confirmed non-AGN / galaxies"
)

plt.scatter(
    umap_df.loc[mask_agn, "UMAP1"],
    umap_df.loc[mask_agn, "UMAP2"],
    s=5,
    alpha=0.75,
    color="tab:red",
    label="Confirmed AGN"
)

plt.scatter(
    umap_df.loc[candidate_mask, "UMAP1"],
    umap_df.loc[candidate_mask, "UMAP2"],
    s=5,
    alpha=0.75,
    color="tab:orange",
    label="AGN candidates from ML"
)

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP: confirmed AGN, confirmed non-AGN, and unlabeled sources")
plt.legend(markerscale=2)
plt.legend(fontsize=10)
plt.show()

In [ ]:
df["xray_opt_r_agn"] = df["xray_opt_r_relation"] >= q80_r
for label_col in ["mateos_wedge", "xray_opt_r_agn"]:
    if label_col in df.columns:
        label_values = df[label_col].fillna(False).astype(bool)
        plt.figure(figsize=(9, 8))

        plt.scatter(
            umap_df.loc[mask_unlabeled, "UMAP1"],
            umap_df.loc[mask_unlabeled, "UMAP2"],
            s=2,
            alpha=0.15,
            color="lightgray",
            label="Unlabeled"
        )

        plt.scatter(
            umap_df.loc[mask_nonagn, "UMAP1"],
            umap_df.loc[mask_nonagn, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:blue",
            label="Confirmed non-AGN / galaxies"
        )

        plt.scatter(
            umap_df.loc[mask_agn, "UMAP1"],
            umap_df.loc[mask_agn, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:red",
            label="Confirmed non-AGN / galaxies"
        )

        plt.scatter(
            umap_df.loc[label_values, "UMAP1"],
            umap_df.loc[label_values, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:orange",
            label=f"{label_col} AGN"
        )

# variability flag to add (in infrared); 
# do corrections for formulas;
# ;

        plt.xlabel("UMAP1")
        plt.ylabel("UMAP2")
        plt.title("UMAP: confirmed AGN, confirmed non-AGN, and unlabeled sources")
        plt.legend(markerscale=2)
        plt.legend(fontsize=10)
        plt.show()

In [ ]:
for label_col in ["xray_agn", "optical_agn", "ir_agn"]:
    if label_col in df.columns:
        label_values = df[label_col].fillna(False).astype(bool)
        plt.figure(figsize=(9, 8))

        plt.scatter(
            umap_df.loc[mask_unlabeled, "UMAP1"],
            umap_df.loc[mask_unlabeled, "UMAP2"],
            s=2,
            alpha=0.15,
            color="lightgray",
            label="Unlabeled"
        )

        plt.scatter(
            umap_df.loc[mask_nonagn, "UMAP1"],
            umap_df.loc[mask_nonagn, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:blue",
            label="Confirmed non-AGN / galaxies"
        )

        plt.scatter(
            umap_df.loc[label_values, "UMAP1"],
            umap_df.loc[label_values, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:red",
            label=f"{label_col} AGN"
        )

        plt.scatter(
            umap_df.loc[mask_silver, "UMAP1"],
            umap_df.loc[mask_silver, "UMAP2"],
            s=5,
            alpha=0.75,
            color="tab:orange",
            label="Objects from silver sample"
        )  

# Take a look check different colour luminoscity: 10^42 -- 10^43 at WISE colour colour diagram. See where they are falling,
# if they are on the side.



        plt.xlabel("UMAP1")
        plt.ylabel("UMAP2")
        plt.title("UMAP: confirmed AGN, confirmed non-AGN, and unlabeled sources")
        plt.legend(markerscale=2)
        plt.legend(fontsize=10)
        plt.show()

In [ ]:
labeled_mask = umap_df["label_3class"].isin([0, 1])

plt.figure(figsize=(9, 7))

plt.scatter(
    umap_df.loc[mask_nonagn, "UMAP1"],
    umap_df.loc[mask_nonagn, "UMAP2"],
    s=6,
    alpha=0.7,
    color="tab:blue",
    label="Confirmed non-AGN / galaxies"
)

plt.scatter(
    umap_df.loc[mask_agn, "UMAP1"],
    umap_df.loc[mask_agn, "UMAP2"],
    s=6,
    alpha=0.7,
    color="tab:red",
    label="Confirmed AGN"
)

plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP labeled subset: AGN vs non-AGN")
plt.legend(markerscale=4)
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    umap_df["UMAP1"],
    umap_df["UMAP2"],
    c=umap_df["NaN_percent"],
    s=2,
    alpha=0.6,
    cmap="viridis"
)

plt.colorbar(label="NaN percent per row")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("UMAP colored by missing-value percentage")
plt.show()

# Machine Learning Part

It is divided into 2 approaches, since we have ~30K of labeled observations and ~140K of unlabeled (where we mostly expect to see AGNs): 
1) Supervised -- we train model on 30K observations and use it to predict 
2) Unsupervised -- we use unlabeled observations for self-training/propagation and increase model's performance

# Training on reduced dataset

We use only the mostly covered columns (11-dimensional space)

In [ ]:
display(df_train.shape)

df_train['combo_g']=df_train['PS21P_g'].combine_first(df_train['LEG21P_G'])
df_train['combo_r']=df_train['PS21P_r'].combine_first(df_train['LEG21P_R'])
df_train['combo_z_mag']=df_train['PS21P_z'].combine_first(df_train['LEG21P_Z'])


df_train = df_train.dropna(subset=['significance', 'flux_aper_b', 'flux_aper_lolim_b', 'flux_aper_hilim_b', 'flux_aper_h','flux_aper_lolim_h',
'flux_aper_hilim_h','flux_aper_s','flux_aper_lolim_s','flux_aper_hilim_s','flux_aper_m','flux_aper_lolim_m','flux_aper_hilim_m','combo_g','combo_r','combo_z_mag','WISE_w1mpro', 
'WISE_w1snr','WISE_w2mpro', 'WISE_w2snr','WISE_w3mpro', 'WISE_w3snr'])

display(df_train.shape)

In [ ]:
df_train["binary_target"] = np.select(
    [
        df_train["label_3class"] == 2,
        df_train["label_3class"] == 0,
    ],
    [
        1,
        0,
    ],
    default=-1
).astype("int8")

# Supervised learning 
df_labeled = df_train[df_train["binary_target"].isin([0, 1])].copy()
y_labeled = df_labeled["binary_target"].astype("int8").to_numpy()

LABEL_COL = "binary_target"

positive_mask = df_train[LABEL_COL] == 1
negative_mask = df_train[LABEL_COL] == 0
unlabeled_mask = df_train[LABEL_COL] == -1

print("Confirmed AGN:", positive_mask.sum())
print("Confirmed non-AGN:", negative_mask.sum())
print("Unlabeled:", unlabeled_mask.sum())

X_aug = df_train.copy()

# Regular Training Procedure

We use initial approach for training on gold sample, with imputation of NaN values, 100% missing feature cut-off, using robustly confirmed labels.

In [ ]:
# Data Cleaning & Pre-processing
feature_missing_percent_train = df_train.isna().mean() * 100
display(feature_missing_percent_train.sort_values(ascending=False).head(20))

max_feature_missing_train = 100

keep_cols_train = feature_missing_percent_train[feature_missing_percent_train < max_feature_missing_train].index
X_train = df_train[keep_cols_train].copy()

X_train.shape

In [ ]:
# Imputing and encoding missing data
from sklearn.impute import SimpleImputer

sample_size = len(df_train) # we can adjust however we want

df_sample = df_train.sample(
    n=min(sample_size, len(df_train)),
    random_state=42
).copy()

X_train = X.loc[df_sample.index].copy()

print(df_sample.shape)
print(X_train.shape)

missing_indicators = X_train.isna().astype("int8")
missing_indicators.columns = [f"{col}_missing" for col in X_train.columns]

imputer = SimpleImputer(strategy="median")
X_clean_train = imputer.fit_transform(X_train)

X_umap_input = np.hstack([
    X_imputed,
    missing_indicators.values
]).astype("float32")

print(X_umap_input.shape)

In [ ]:
df_train["binary_target"] = np.select(
    [
        df_train["label_3class"] == 2,
        df_train["label_3class"] == 0,
    ],
    [
        1,
        0,
    ],
    default=-1
).astype("int8")

# Supervised learning 
df_labeled = df_train[df_train["binary_target"].isin([0, 1])].copy()
y_labeled = df_labeled["binary_target"].astype("int8").to_numpy()

LABEL_COL = "binary_target"

positive_mask = df_train[LABEL_COL] == 1
negative_mask = df_train[LABEL_COL] == 0
unlabeled_mask = df_train[LABEL_COL] == -1

print("Confirmed AGN:", positive_mask.sum())
print("Confirmed non-AGN:", negative_mask.sum())
print("Unlabeled:", unlabeled_mask.sum())

In [ ]:
metadata_cols = [
    "name",
    "col1",
    
    # AGN labels
    "xray_agn",
    "optical_agn",
    "ir_agn",
    "agn_class",
    "agn_positive",
    
    # non-AGN labels
    "non_agn",
    "agn_negative",
    
    # derived labels
    "label_3class",
    "label_conflict",
    "silver_score",
    "silver_agn",
    "binary_target",
    
    # training info
    "train_label", "train_weight", "label_source", "sample_weight",
    
    # masks
    "mateos_wedge",
    "wise_stern",

    # class / metadata columns
    "likelihood_class",
    "SDSS_CLASS",
    "SDSS_SUBCLASS",
    "2MASS21P_designation"
]

coords = ["ra", "dec", "gal_l", "gal_b"]


# Model may learn sky coverage -> do an experiment with coords and without coords
exclude_cols_train = [c for c in metadata_cols + coords if c in df_train.columns]


# Drop columns and select numerical features
X_all = df_train.drop(columns=exclude_cols_train, errors="ignore")
print(X_all.shape)

X_all = X_all.select_dtypes(include=[np.number, "bool"]).copy()
print(X_all.shape)

X_all = X_all.replace([np.inf, -np.inf], np.nan)

display(X_all.head(20))


In [ ]:
# Adding missing flags as indicators
missing_flags = X_all.isna().astype("int8")
missing_flags.columns = [f"{c}_missing" for c in X_all.columns]

X_aug = pd.concat([X_all, missing_flags], axis=1)

X_aug.shape

In [ ]:
# Small check for other random-labeled bad columns
bad_keywords = [
    "redshift", "spec", "spectro",
    "class", "subclass",
    "label",
    "likelihood", "prob"
]

clean_cols = [
    c for c in X_aug.columns
    if not any(k in c.lower() for k in bad_keywords)
]


X_aug_clean = X_aug[clean_cols]

X_aug = X_aug_clean.copy()

# Common ML training algorithm

In [ ]:
# ML here
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)
from sklearn.base import clone

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Labeled
labeled_mask = df_train[LABEL_COL].isin([0, 1])

X_labeled = X_aug.loc[labeled_mask].copy()
y_labeled = (
    df_train.loc[labeled_mask, LABEL_COL]
    .astype("int8")
    .to_numpy()
)

print(X_labeled.shape)
print(pd.Series(y_labeled).value_counts())

In [ ]:
# Models
base_models = {
    "lightgbm": LGBMClassifier(
        n_estimators=700,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    ),

    "catboost": CatBoostClassifier(
        iterations=700,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=5,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    ),
}

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

unlabeled_idx = df_train.index[
    df_train[LABEL_COL] == -1
]

X_unlabeled = X_aug.loc[unlabeled_idx]

oof_predictions = {}
unlabeled_fold_predictions = {}

for model_name, base_model in base_models.items():
    print(f"\nMODEL: {model_name}")

    oof = np.zeros(len(X_labeled))
    u_fold_preds = []

    for fold, (train_pos, valid_pos) in enumerate(
        cv.split(X_labeled, y_labeled),
        start=1,
    ):
        model = clone(base_model)

        X_tr = X_labeled.iloc[train_pos]
        X_va = X_labeled.iloc[valid_pos]

        y_tr = y_labeled[train_pos]
        y_va = y_labeled[valid_pos]

        model.fit(X_tr, y_tr)

        valid_pred = model.predict_proba(X_va)[:, 1]
        oof[valid_pos] = valid_pred

        unlabeled_pred = model.predict_proba(X_unlabeled)[:, 1]
        u_fold_preds.append(unlabeled_pred)

        print(
            f"Fold {fold}:",
            "ROC-AUC =", round(
                roc_auc_score(y_va, valid_pred), 5
            ),
            "PR-AUC =", round(
                average_precision_score(y_va, valid_pred), 5
            ),
        )

    oof_predictions[model_name] = oof
    unlabeled_fold_predictions[model_name] = np.vstack(
        u_fold_preds
    )

    print(
        "OOF ROC-AUC:",
        roc_auc_score(y_labeled, oof)
    )
    print(
        "OOF PR-AUC:",
        average_precision_score(y_labeled, oof)
    )

In [ ]:
from scipy.special import expit, logit

eps = 1e-6

lgbm_u = unlabeled_fold_predictions["lightgbm"]
cat_u = unlabeled_fold_predictions["catboost"]

# Средний прогноз каждого алгоритма по folds
lgbm_mean = lgbm_u.mean(axis=0)
cat_mean = cat_u.mean(axis=0)

# Ensemble
ensemble_score = expit(
    (
        logit(np.clip(lgbm_mean, eps, 1 - eps))
        +
        logit(np.clip(cat_mean, eps, 1 - eps))
    ) / 2
)

# Неопределённость между folds и моделями
all_u_predictions = np.vstack([
    lgbm_u,
    cat_u,
])

ensemble_std = all_u_predictions.std(axis=0)

df_train.loc[
    unlabeled_idx,
    "score_stage1"
] = ensemble_score

df_train.loc[
    unlabeled_idx,
    "score_stage1_std"
] = ensemble_std

In [ ]:
u_results = pd.DataFrame(
    {
        "lgbm_score": lgbm_mean,
        "cat_score": cat_mean,
        "ensemble_score": ensemble_score,
        "score_std": ensemble_std,
    },
    index=unlabeled_idx,
)

u_results["model_disagreement"] = (
    u_results["lgbm_score"]
    - u_results["cat_score"]
).abs()

display(u_results.describe())

In [ ]:
pseudo_positive_fraction = 0.20

score_cutoff = u_results["ensemble_score"].quantile(
    1 - pseudo_positive_fraction
)

std_cutoff = u_results["score_std"].quantile(0.50)
disagreement_cutoff = u_results[
    "model_disagreement"
].quantile(0.50)

pseudo_positive_mask = (
    (u_results["ensemble_score"] >= score_cutoff)
    &
    (u_results["score_std"] <= std_cutoff)
    &
    (
        u_results["model_disagreement"]
        <= disagreement_cutoff
    )
)

pseudo_positive_idx = u_results.index[
    pseudo_positive_mask
]

print("Pseudo-positive candidates:", len(pseudo_positive_idx))

In [ ]:
def pseudo_positive_bagging(
    X,
    labels,
    candidate_positive_idx,
    base_model,
    n_bags=30,
    candidate_fraction=0.7,
    pseudo_weight=0.2,
    seed=42,
):
    rng = np.random.default_rng(seed)

    positive_idx = np.flatnonzero(labels == 1)
    negative_idx = np.flatnonzero(labels == 0)

    candidate_positions = X.index.get_indexer(
        candidate_positive_idx
    )
    candidate_positions = candidate_positions[
        candidate_positions >= 0
    ]

    predictions = []

    for bag in range(n_bags):
        # Bootstrap confirmed labels
        bag_pos = rng.choice(
            positive_idx,
            size=len(positive_idx),
            replace=True
        )

        bag_neg = rng.choice(
            negative_idx,
            size=len(negative_idx),
            replace=True
        )

        n_candidates = max(
            1,
            int(
                len(candidate_positions)
                * candidate_fraction
            )
        )

        bag_pseudo = rng.choice(
            candidate_positions,
            size=n_candidates,
            replace=False
        )

        train_positions = np.concatenate([
            bag_pos,
            bag_neg,
            bag_pseudo,
        ])

        y_train = np.concatenate([
            np.ones(len(bag_pos), dtype="int8"),
            np.zeros(len(bag_neg), dtype="int8"),
            np.ones(len(bag_pseudo), dtype="int8"),
        ])

        sample_weight = np.concatenate([
            np.ones(len(bag_pos)),
            np.ones(len(bag_neg)),
            np.full(len(bag_pseudo), pseudo_weight),
        ])

        model = clone(base_model)

        if model.__class__.__name__.startswith("LGBM"):
            model.set_params(random_state=seed + bag)

        model.fit(
            X.iloc[train_positions],
            y_train,
            sample_weight=sample_weight,
        )

        predictions.append(
            model.predict_proba(X)[:, 1]
        )

    predictions = np.vstack(predictions)

    return (
        predictions.mean(axis=0),
        predictions.std(axis=0),
        predictions,
    )

In [ ]:
labels_all = df_train[LABEL_COL].to_numpy()

bag_score_lgbm, bag_std_lgbm, _ = (
    pseudo_positive_bagging(
        X=X_aug,
        labels=labels_all,
        candidate_positive_idx=pseudo_positive_idx,
        base_model=base_models["lightgbm"],
        n_bags=30,
        candidate_fraction=0.7,
        pseudo_weight=0.20,
        seed=42,
    )
)

df_train["score_bagged_pseudo_lgbm"] = (
    bag_score_lgbm
)

df_train["score_bagged_pseudo_lgbm_std"] = (
    bag_std_lgbm
)

In [ ]:
candidate_mask = df_train["score_bagged_pseudo_lgbm"] > 0.8

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.base import clone
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import numpy as np

train_mask = df["train_label"].isin([0, 1])
df_sup = df.loc[train_mask].copy()

X = df_sup[feature_cols]
y = df_sup["train_label"].astype(int)
w = df_sup["train_weight"].astype(float)

models = {
    "lgbm": LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.015,
        num_leaves=31,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=5.0,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),
    "cat": CatBoostClassifier(
        iterations=1500,
        learning_rate=0.02,
        depth=6,
        l2_leaf_reg=8,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = {}

for name, base_model in models.items():
    print("\nMODEL:", name)
    oof = np.zeros(len(df_sup))

    for fold, (tr, va) in enumerate(cv.split(X, y), 1):
        model = clone(base_model)

        model.fit(
            X.iloc[tr], y.iloc[tr],
            sample_weight=w.iloc[tr]
        )

        pred = model.predict_proba(X.iloc[va])[:, 1]
        oof[va] = pred

        print(
            fold,
            "ROC:", round(roc_auc_score(y.iloc[va], pred), 4),
            "PR:", round(average_precision_score(y.iloc[va], pred), 4),
            "Brier:", round(brier_score_loss(y.iloc[va], pred), 4)
        )

    oof_preds[name] = oof
    print("OOF ROC:", round(roc_auc_score(y, oof), 4))
    print("OOF PR:", round(average_precision_score(y, oof), 4))

In [ ]:
missing_cols = [c for c in X_aug.columns if c.endswith("_missing")]
X_missing_only = X_aug[missing_cols]

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

X_labeled = X_aug.loc[df_labeled.index].copy()

# Подготовка таргета: сбрасываем индексы и переводим в numpy-массив
y_labeled_arr = y_labeled.values if hasattr(y_labeled, 'values') else np.array(y_labeled)
X_labeled_reset = X_labeled.reset_index(drop=True)

unique_classes = np.unique(y_labeled_arr)
n_classes = len(unique_classes) # Будет равно 3

models = {
    "lightgbm": LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        class_weight="balanced", # Отлично работает и для мультикласса
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    ),
    "catboost": CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=5,
        loss_function="MultiClass",     # <-- ИЗМЕНЕНО: Для 3 классов вместо Logloss
        eval_metric="MultiClass",       # <-- ИЗМЕНЕНО: AUC в мультиклассе на валидации ведет себя капризно
        auto_class_weights="Balanced",  # Поддерживает балансировку мультикласса
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Создаем oof-массив размерностью (количество_строк, 3_класса)
# Каждая ячейка будет хранить матрицу вероятностей для каждого класса
oof = {name: np.zeros((len(df_labeled), n_classes)) for name in models}

for name, base_model in models.items():
    print("\nMODEL:", name)

    for fold, (tr, va) in enumerate(cv.split(X_labeled_reset, y_labeled_arr), 1):
        X_tr, X_va = X_labeled_reset.iloc[tr], X_labeled_reset.iloc[va]
        y_tr, y_va = y_labeled_arr[tr], y_labeled_arr[va]

        # Создаем чистую копию модели, чтобы веса не накапливались от фолда к фолду
        model = clone(base_model)
        model.fit(X_tr, y_tr)
        
        # Получаем матрицу вероятностей размера (len(va), 3)
        pred = model.predict_proba(X_va)
        oof[name][va] = pred

        # Считаем ROC-AUC по стратегии One-vs-Rest (Один против всех)
        fold_roc = roc_auc_score(y_va, pred, multi_class='ovr', labels=unique_classes)
        
        # PR-AUC (Average Precision) из коробки не поддерживает мультикласс.
        # Считаем среднее бинарных PR-AUC для каждого из 3-х классов.
        fold_pr = np.mean([
            average_precision_score((y_va == c).astype(int), pred[:, i]) 
            for i, c in enumerate(unique_classes)
        ])

        print(f"fold {fold} | ROC-AUC (OVR): {round(fold_roc, 4)} | PR-AUC (Macro): {round(fold_pr, 4)}")

    # Итоговый скор по всей выборке (Out-of-Fold)
    oof_roc = roc_auc_score(y_labeled_arr, oof[name], multi_class='ovr', labels=unique_classes)
    oof_pr = np.mean([
        average_precision_score((y_labeled_arr == c).astype(int), oof[name][:, i]) 
        for i, c in enumerate(unique_classes)
    ])
    
    print(f"--- FINAL OOF ROC-AUC: {round(oof_roc, 4)}")
    print(f"--- FINAL OOF PR-AUC: {round(oof_pr, 4)}")


In [ ]:
# CatBoost / LightGBM for BINARY Classes
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
import numpy as np

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


X_labeled = X_aug.loc[df_labeled.index].copy()

models = {
    "lightgbm": LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    ),
    "catboost": CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=5,
        loss_function="Logloss",
        eval_metric="AUC",
        auto_class_weights="Balanced",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = {name: np.zeros(len(df_labeled)) for name in models}

for name, model in models.items():
    print("\nMODEL:", name)

    for fold, (tr, va) in enumerate(cv.split(X_labeled, y_labeled), 1):
        X_tr, X_va = X_labeled.iloc[tr], X_labeled.iloc[va]
        y_tr, y_va = y_labeled[tr], y_labeled[va]

        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_va)[:, 1]
        oof[name][va] = pred

        print(
            "fold", fold,
            "ROC-AUC", round(roc_auc_score(y_va, pred), 4),
            "PR-AUC", round(average_precision_score(y_va, pred), 4)
        )

    print("OOF ROC-AUC:", roc_auc_score(y_labeled, oof[name]))
    print("OOF PR-AUC:", average_precision_score(y_labeled, oof[name]))

In [ ]:
df_unknown = df[df["label_3class"] == -1].copy()

X_unknown = X_aug.loc[df_unknown.index].copy()

# LightGBM
final_lgbm = models["lightgbm"]
final_lgbm.fit(X_labeled, y_labeled)

unknown_score_lgbm = final_lgbm.predict_proba(X_unknown)[:, 1]

# CatBoost
final_cat = models["catboost"]
final_cat.fit(X_labeled, y_labeled)

unknown_score_cat = final_cat.predict_proba(X_unknown)[:, 1]


# Defragmentate memory
df = df.copy()

df.loc[df_unknown.index, "score_supervised_lgbm"] = unknown_score_lgbm
df.loc[df_unknown.index, "score_supervised_cat"] = unknown_score_cat
df.loc[df_unknown.index, "score_supervised_mean"] = (unknown_score_lgbm + unknown_score_cat) / 2

In [ ]:
unknown_mask = df["label_3class"] == -1

display((df.loc[unknown_mask, "score_supervised_lgbm"] > 0.7).value_counts())
display((df.loc[unknown_mask, "score_supervised_cat"] > 0.7).value_counts())

check_mask = (
    unknown_mask
    & ((df["score_supervised_lgbm"] > 0.7)
    | (df["score_supervised_cat"] > 0.7))
)

display(len(df.loc[check_mask]))

In [ ]:
# Multi-iterative model

final_lgbm = models["lightgbm"]
final_cat = models["catboost"]

for i in range(0, 10):
    # LightGBM
    final_lgbm.fit(X_labeled, y_labeled)
    unknown_score_lgbm = final_lgbm.predict_proba(X_unknown)[:, 1]

    # CatBoost
    final_cat.fit(X_labeled, y_labeled)
    unknown_score_cat = final_cat.predict_proba(X_unknown)[:, 1]


    # Defragmentate memory
    X_labeled = df.copy()

    X_labeled.loc[df_unknown.index, "score_supervised_lgbm"] = unknown_score_lgbm
    X_labeled.loc[df_unknown.index, "score_supervised_cat"] = unknown_score_cat
    X_labeled.loc[df_unknown.index, "score_supervised_mean"] = (unknown_score_lgbm + unknown_score_cat) / 2
        

In [ ]:
df.groupby("label_3class")["score_supervised_lgbm"].describe()

In [ ]:
# PNU Learning 

from sklearn.base import clone

def bagging_pnu_scores(
    X_aug,
    labels,
    base_model,
    n_bags=30,
    unlabeled_ratio=1.0,
    unlabeled_weight=0.25,
    seed=42,
):
    rng = np.random.default_rng(seed)

    pos_idx = np.flatnonzero(labels == 1)
    neg_idx = np.flatnonzero(labels == 0)
    unl_idx = np.flatnonzero(labels == -1)

    score_sum = np.zeros(len(labels), dtype="float64")
    score_sq = np.zeros(len(labels), dtype="float64")
    score_count = np.zeros(len(labels), dtype="int32")

    n_temp_u = min(
        int((len(pos_idx) + len(neg_idx)) * unlabeled_ratio),
        len(unl_idx)
    )

    for b in range(n_bags):
        temp_u = rng.choice(unl_idx, size=n_temp_u, replace=False)

        train_idx = np.concatenate([pos_idx, neg_idx, temp_u])

        y_train = np.concatenate([
            np.ones(len(pos_idx), dtype=int),
            np.zeros(len(neg_idx), dtype=int),
            np.zeros(len(temp_u), dtype=int),
        ])

        sample_weight = np.concatenate([
            np.ones(len(pos_idx)),
            np.ones(len(neg_idx)),
            np.full(len(temp_u), unlabeled_weight),
        ])

        model = clone(base_model)
        if hasattr(model, "set_params") and "random_state" in model.get_params():
            model.set_params(random_state=seed + b)

        model.fit(
            X_aug.iloc[train_idx],
            y_train,
            sample_weight=sample_weight
        )

        pred = model.predict_proba(X_aug)[:, 1]

        # Для unlabeled делаем honest/OOB:
        # не считаем прогноз для тех unlabeled, которые были временными negatives
        honest = np.ones(len(labels), dtype=bool)
        honest[temp_u] = False

        score_sum[honest] += pred[honest]
        score_sq[honest] += pred[honest] ** 2
        score_count[honest] += 1

    valid = score_count > 0

    mean_score = np.full(len(labels), np.nan)
    std_score = np.full(len(labels), np.nan)

    mean_score[valid] = score_sum[valid] / score_count[valid]

    var = score_sq[valid] / score_count[valid] - mean_score[valid] ** 2
    std_score[valid] = np.sqrt(np.maximum(var, 0))

    return mean_score, std_score, score_count

In [ ]:
labels_all = df["label_3class"].to_numpy()

base_lgbm = LGBMClassifier(
    n_estimators=700,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=2.0,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

score_pnu, score_pnu_std, score_pnu_count = bagging_pnu_scores(
    X_aug=X_aug,
    labels=labels_all,
    base_model=base_lgbm,
    n_bags=30,
    unlabeled_ratio=0.5,
    unlabeled_weight=0.15,
    seed=42,
)

df["score_pnu_lgbm"] = score_pnu
df["score_pnu_lgbm_std"] = score_pnu_std
df["score_pnu_lgbm_count"] = score_pnu_count

In [ ]:
display((df["score_pnu_lgbm"] > 0.99).value_counts())

df["score_pnu_lgbm"].describe()

In [ ]:
df.groupby("label_3class")["score_pnu_lgbm"].describe()

In [ ]:
pseudo_pos_mask = (
    (df["label_3class"] == -1)
    & (df["score_pnu_lgbm"] > 0.95)
    & (df["score_pnu_lgbm_std"] < 0.05)
)

pseudo_neg_mask = (
    (df["label_3class"] == -1)
    & (df["score_pnu_lgbm"] < 0.05)
    & (df["score_pnu_lgbm_std"] < 0.05)
)

In [ ]:
# Unsupervised learning

y_ssl = df["label_3class"].copy()

print(df["label_3class"].value_counts().sort_index())
